In [10]:

import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [11]:

train = pd.read_csv("vehicle_train_10000.csv")
test = pd.read_csv("vehicle_test.csv")

print("Training dataset shape:", train.shape)
print("Test dataset shape:", test.shape)

print("\nFirst five training rows:")
display(train.head())

print("\nVehicle classes:")
print(train["vehicle_class"].value_counts().sort_index())


Training dataset shape: (10000, 22)
Test dataset shape: (2500, 21)

First five training rows:


,vehicle_id,engine_displacement_l,horsepower_hp,vehicle_weight_kg,wheelbase_mm,vehicle_length_mm,vehicle_width_mm,vehicle_height_mm,fuel_tank_capacity_l,city_fuel_efficiency_kmpl,...,torque_nm,ground_clearance_mm,turning_radius_m,cargo_capacity_l,acceleration_0_100_kmh_s,braking_distance_m,tire_width_mm,drag_coefficient,battery_capacity_kwh,vehicle_class
0,1,2.56,290.0,1532.0,2737.0,4538.0,1814.0,1544.0,65.0,9.69,...,438.0,197.0,6.05,656.0,5.87,43.0,267.0,0.34,79.0,2
1,2,2.70,350.0,1529.0,2726.0,4473.0,1848.0,1636.0,62.0,10.35,...,441.0,251.0,5.30,513.0,6.48,41.0,247.0,0.28,76.0,2
2,3,2.50,335.0,1100.0,2504.0,3900.0,1650.0,1251.0,35.0,13.70,...,471.0,110.0,5.58,150.0,3.90,29.0,261.0,0.23,89.0,2
3,4,3.00,345.0,2069.0,3053.0,5134.0,2036.0,1572.0,81.0,10.72,...,478.0,168.0,6.36,626.0,6.20,51.0,283.0,0.37,60.0,1
4,5,1.59,155.0,1574.0,2791.0,4678.0,1804.0,1492.0,60.0,13.55,...,235.0,179.0,5.43,447.0,10.15,47.0,201.0,0.34,43.0,1



Vehicle classes:
vehicle_class
0    3389
1    3326
2    3285
Name: count, dtype: int64


In [12]:

# Separate the input features and target labels
X = train.drop(columns=["vehicle_id", "vehicle_class"])
y = train["vehicle_class"]

# Test data has no target column
X_test = test.drop(columns=["vehicle_id"])

print("Number of input features:", X.shape[1])
print("Target classes:", sorted(y.unique()))


Number of input features: 20
Target classes: [np.int64(0), np.int64(1), np.int64(2)]


In [13]:

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Build a pipeline: fill missing values, scale features,
# then train Logistic Regression.
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=3000,
        C=1.0,
        solver="lbfgs"
    )
)

# Train the model
model.fit(X_train, y_train)

# Predict the validation labels
y_pred = model.predict(X_valid)

# Evaluate the model
accuracy = accuracy_score(y_valid, y_pred)

print(f"Validation accuracy: {accuracy:.4f}")
print(f"Validation accuracy: {accuracy * 100:.2f}%")

print("\nClassification report:")
print(classification_report(y_valid, y_pred))


Validation accuracy: 0.8835
Validation accuracy: 88.35%

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       678
           1       0.89      0.90      0.90       665
           2       0.93      0.92      0.92       657

    accuracy                           0.88      2000
   macro avg       0.88      0.88      0.88      2000
weighted avg       0.88      0.88      0.88      2000



In [14]:

# Retrain using the complete training dataset
model.fit(X, y)

# Predict the vehicle class for every test row
test_predictions = model.predict(X_test)

# Create the required submission file
submission = pd.DataFrame({
    "vehicle_id": test["vehicle_id"],
    "vehicle_class": test_predictions.astype(int)
})

# Save as CSV
submission.to_csv("vehicle_submission.csv", index=False)

print(submission.head())
print("\nSubmission shape:", submission.shape)
print("\nPredicted class counts:")
print(submission["vehicle_class"].value_counts().sort_index())

# Download the submission
files.download("vehicle_submission.csv")


   vehicle_id  vehicle_class
0       10001              0
1       10002              0
2       10003              1
3       10004              2
4       10005              1

Submission shape: (2500, 2)

Predicted class counts:
vehicle_class
0    837
1    833
2    830
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>